In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")

In [3]:
import sys

#仓库根：往上找含 config.py 的目录（本 notebook 就在仓库里，换机器不用改）
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config

In [4]:
SEED = 114514
N_FOLD = 10
N_MORDRED_TOP = 120
BAG_SEEDS = [114514, 2024, 777]
SPLIT_SEEDS = [114514, 7917, 323922, 20261029, 20261030]
SEEDS_811 = [114514, 7917, 323922, 20261029, 20261030]
SEEDS_721 = [114514, 7917, 323922]
N_CANDS = [80, 120, 160, 200]

In [5]:
OUT = config.OUT      #产物目录 TDT/out/：features_1426.npz 读这里，CSV/NPZ 也落这里

#只用来“排前 120”的轻量模型
EMBED_PARAMS = dict(n_estimators = 100, learning_rate = 0.05, max_depth = 6,
                    reg_lambda = 1.0, n_jobs = 8, random_state = SEED,
                    importance_type = "gain")

#ens4 的四个成员
XGB_PARAMS = dict(n_estimators = 800, learning_rate = 0.05, max_depth = 6, reg_lambda = 3.0,
                  min_child_weight = 1, subsample = 0.7, colsample_bytree = 0.6,
                  n_jobs = 8, random_state = SEED)
ET_PARAMS = dict(n_estimators = 500, n_jobs = 8, random_state = SEED)
HIST_PARAMS = dict(max_iter = 500, learning_rate = 0.05, max_leaf_nodes = 31,
                   l2_regularization = 1.0, random_state = SEED)
MLP_PARAMS = dict(hidden_layer_sizes = (64, 32), max_iter = 600, alpha = 1e-3,
                  early_stopping = True, random_state = SEED)

In [6]:
def filter_features(df):
    # 筛除方差为 0 的列 + |corr| > 0.9 的冗余列
    vt = VarianceThreshold(threshold = 0).fit(df)
    d = pd.DataFrame(vt.transform(df), columns = df.columns[vt.get_support()])
    corr = d.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
    drop = [c for c in upper.columns if any(upper[c] > 0.9)]
    return d.drop(columns = drop)

def select_top(tr, mmat, mcols, y, n = N_MORDRED_TOP):
    #只用训练折：过滤 → XGB(gain) 前 n，返回列名
    d = filter_features(pd.DataFrame(mmat[tr], columns = mcols))
    xgb = XGBRegressor(**EMBED_PARAMS)
    imp = pd.Series(xgb.fit(d.values, y[tr]).feature_importances_, index = d.columns).sort_values(ascending = False)
    return list(imp.index[ : n])

def build_x(mmat, qx, names, mcols):
    #选出的 Mordred 列 + 4 个量化列。量化列的 NaN 保留（XGB/ET/HistGB 原生处理）
    cols = [mcols.index(c) for c in names]
    return np.column_stack([mmat[:, cols], qx])

def ens4(xtr, ytr, xte):
    #四个模型各训一遍，预测取平均。MLP 不能吃 NaN，用 pipeline 里的中位数填补
    xgb = XGBRegressor(**XGB_PARAMS).fit(xtr, ytr).predict(xte)
    et = ExtraTreesRegressor(**ET_PARAMS).fit(xtr, ytr).predict(xte)
    hist = HistGradientBoostingRegressor(**HIST_PARAMS).fit(xtr, ytr).predict(xte)
    mlp = make_pipeline(SimpleImputer(strategy = "median"), StandardScaler(),
                        MLPRegressor(**MLP_PARAMS)).fit(xtr, ytr).predict(xte)
    return np.mean([xgb, et, hist, mlp], axis = 0)

def metrics(y, pred, gid):
    #整体 R²、MAE、组内 R²（按分子式分组）、组间 R²（只有一组时记 NaN）
    total = ((y - y.mean()) ** 2).sum()
    d = pd.DataFrame({"g": gid, "y": y, "p": pred, "r": y - pred})
    w_tot = d.groupby("g")["y"].apply(lambda s: ((s - s.mean()) ** 2).sum()).sum()
    w_res = d.groupby("g")["r"].apply(lambda s: ((s - s.mean()) ** 2).sum()).sum()
    gmean = d.groupby("g")["y"].transform("mean")
    pmean = d.groupby("g")["p"].transform("mean")
    b_tot = total - w_tot
    b_res = ((pmean - gmean) ** 2).sum()
    return dict(R2 = round(float(r2_score(y, pred)), 4),
                MAE = round(float(mean_absolute_error(y, pred)), 2),
                组内R2 = round(float(1 - w_res / w_tot), 4) if w_tot > 1e-12 else float("nan"),
                组间R2 = round(float(1 - b_res / b_tot), 4) if b_tot > 1e-12 else float("nan"))

In [7]:
# 数据 + 两个分组：分子式（组内 R²）与 Murcko 骨架（骨架划分，无环分子记“无环”）
npz = np.load(OUT / "features_1426.npz", allow_pickle = True)
mmat = npz["mmat"].astype(np.float32)
mcols = [str(c) for c in npz["mcols"]]
qx = npz["qx"].astype(float)
y = npz["y"].astype(float)

bad_cols = np.where(np.isnan(qx).all(axis = 0))[0].tolist()
if bad_cols:
    raise ValueError(
        f"features_1426.npz 的量化列 {bad_cols} 全是 NaN。"
        "先跑 Features_Producer，或检查 CHEM_PARENTS / CHEM_XTB_WORK 是否指对。"
    )

df = pd.read_excel(config.DATA_TDT / "fixed.xlsx")
df.columns = ["Td", "SMILES"]
gid = np.array([rdMolDescriptors.CalcMolFormula(Chem.MolFromSmiles(s)) for s in df["SMILES"]])
scaf = []
for s in df["SMILES"]:
    try:
        scaf.append(MurckoScaffold.MurckoScaffoldSmiles(mol = Chem.MolFromSmiles(s)) or "无环")
    except Exception:
        scaf.append("无环")
scaf = np.array(scaf)

print(f"数据 {mmat.shape}，量化列 {qx.shape}，分子式 {len(set(gid))} 组，Murcko 骨架 {len(set(scaf))} 种")

数据 (1426, 1449)，量化列 (1426, 4)，分子式 988 组，Murcko 骨架 560 种


In [18]:
# ① 随机 10 折（折种子 SEED）：每折只用训练折选 120 个特征，测试折各预测一次
oof_seed = np.zeros(len(y))
for k, (tr, te) in enumerate(KFold(N_FOLD, shuffle = True, random_state = SEED).split(mmat)):
    names = select_top(tr, mmat, mcols, y)
    ranks = pd.DataFrame({"fold": k, "rank": np.arange(1, len(names) + 1), "col_name": names})
    ranks.to_csv(OUT / f"selected_features_fold{k}.csv", index = False, encoding = "utf-8-sig")
    x = build_x(mmat, qx, names, mcols)
    oof_seed[te] = ens4(x[tr], y[tr], x[te])

m = metrics(y, oof_seed, gid)
np.savez(OUT / "oof_random_seed114514.npz", oof = oof_seed)
print(f"随机 10 折 seed {SEED}: R2 = {m['R2']:.4f}  MAE = {m['MAE']:.2f}  "
      f"组内 = {m['组内R2']:+.4f}  组间 = {m['组间R2']:+.4f}")
m

随机 10 折 seed 114514: R2 = 0.6996  MAE = 27.18  组内 = +0.4349  组间 = +0.7493


{'R2': 0.6996, 'MAE': 27.18, '组内R2': 0.4349, '组间R2': 0.7493}

In [23]:
# 把 10 折选出的特征表收进子文件夹
d_feat = OUT / "selected_features_folds"
d_feat.mkdir(exist_ok = True)
for p in sorted(OUT.glob("selected_features_fold*.csv")):
    p.replace(d_feat / p.name)
d_feat

WindowsPath('C:/Users/czywg/Desktop/Programmer_czy/chem_project/TDT/out/selected_features_folds')

In [19]:
# ② bag3：再换两个折种子各跑一遍 10 折，三个种子逐分子平均
oofs = [oof_seed]
rows = [dict(配置 = f"随机10折 seed{SEED}", **m)]
for seed in BAG_SEEDS[1 : ]:
    oof = np.zeros(len(y))
    for tr, te in KFold(N_FOLD, shuffle = True, random_state = seed).split(mmat):
        names = select_top(tr, mmat, mcols, y)
        x = build_x(mmat, qx, names, mcols)
        oof[te] = ens4(x[tr], y[tr], x[te])
    mk = metrics(y, oof, gid)
    print(f"随机 10 折 seed {seed}: R2 = {mk['R2']:.4f}  MAE = {mk['MAE']:.2f}  "
          f"组内 = {mk['组内R2']:+.4f}  组间 = {mk['组间R2']:+.4f}")
    rows.append(dict(配置 = f"随机10折 seed{seed}", **mk))
    oofs.append(oof)

bag = np.mean(oofs, axis = 0)
m_bag = metrics(y, bag, gid)
print(f"bag3: R2 = {m_bag['R2']:.4f}  MAE = {m_bag['MAE']:.2f}  "
      f"组内 = {m_bag['组内R2']:+.4f}  组间 = {m_bag['组间R2']:+.4f}")
rows.append(dict(配置 = "随机 bag3", **m_bag))
np.savez(OUT / "oof_bag3.npz", oof = bag)
pd.DataFrame(rows).to_csv(OUT / "metrics.csv", index = False, encoding = "utf-8-sig")
pd.DataFrame(rows)

随机 10 折 seed 2024: R2 = 0.7013  MAE = 27.10  组内 = +0.4323  组间 = +0.7518
随机 10 折 seed 777: R2 = 0.7022  MAE = 27.06  组内 = +0.4468  组间 = +0.7502
bag3: R2 = 0.7073  MAE = 26.81  组内 = +0.4478  组间 = +0.7560


,配置,R2,MAE,组内R2,组间R2
0,随机10折 seed114514,0.6996,27.18,0.4349,0.7493
1,随机10折 seed2024,0.7013,27.10,0.4323,0.7518
2,随机10折 seed777,0.7022,27.06,0.4468,0.7502
3,随机 bag3,0.7073,26.81,0.4478,0.7560


In [20]:
# ③ 骨架 8:2 × 5 种子（skfp 的 randomized_scaffold_train_test_split，整组骨架留出）
from skfp.model_selection import randomized_scaffold_train_test_split
smiles = df["SMILES"].tolist()
assert len(set(smiles)) == len(smiles), "SMILES 有重复，按 SMILES 映射会有歧义"

rows_sc = []
for s in SPLIT_SEEDS:
    a, b, ya, yb = randomized_scaffold_train_test_split(smiles, y, train_size = 0.8, test_size = 0.2, return_indices = True, random_state = s)
    a, b = np.asarray(a), np.asarray(b)
    tr, te = (a, b) if len(a) >= len(b) else (b, a)
    names = select_top(tr, mmat, mcols, y)
    x = build_x(mmat, qx, names, mcols)
    mk = metrics(y[te], ens4(x[tr], y[tr], x[te]), scaf[te])
    rows_sc.append(dict(评估 = "骨架8:2", seed = s, n_test = len(te), **mk))
    print(f"骨架8:2 seed {s}: n_test = {len(te)}  R2 = {mk['R2']:.4f}  MAE = {mk['MAE']:.2f}")

sub = pd.DataFrame([r for r in rows_sc])
print(f"骨架8:2 x5: R2 = {sub['R2'].mean():.4f} +- {sub['R2'].std():.4f}  MAE = {sub['MAE'].mean():.2f}")

骨架8:2 seed 114514: n_test = 320  R2 = 0.6594  MAE = 29.98
骨架8:2 seed 7917: n_test = 288  R2 = 0.6400  MAE = 29.44
骨架8:2 seed 323922: n_test = 286  R2 = 0.6943  MAE = 29.54
骨架8:2 seed 20261029: n_test = 288  R2 = 0.6630  MAE = 31.02
骨架8:2 seed 20261030: n_test = 288  R2 = 0.6229  MAE = 31.73
骨架8:2 x5: R2 = 0.6559 +- 0.0268  MAE = 30.34


In [22]:
# ④ 骨架分组 10 折（GroupKFold：同一骨架的分子整组留出）
r2s = []
for k, (tr, te) in enumerate(GroupKFold(N_FOLD).split(mmat, y, scaf)):
    names = select_top(tr, mmat, mcols, y)
    x = build_x(mmat, qx, names, mcols)
    mk = metrics(y[te], ens4(x[tr], y[tr], x[te]), scaf[te])
    r2s.append(mk["R2"])
    rows_sc.append(dict(评估 = "骨架分组10折", seed = SEED, n_test = len(te), **mk))
    print(f"骨架分组 fold {k}: n_test = {len(te)}  R2 = {mk['R2']:.4f}  MAE = {mk['MAE']:.2f}")
print(f"骨架分组10折: R2 = {np.mean(r2s):.4f} +- {np.std(r2s):.4f}")
pd.DataFrame(rows_sc).to_csv(OUT / "scaffold_metrics.csv", index = False, encoding = "utf-8-sig")

骨架分组 fold 0: n_test = 150  R2 = 0.2281  MAE = 25.19
骨架分组 fold 1: n_test = 142  R2 = 0.6057  MAE = 29.11
骨架分组 fold 2: n_test = 142  R2 = 0.6097  MAE = 31.45
骨架分组 fold 3: n_test = 142  R2 = 0.5935  MAE = 35.16
骨架分组 fold 4: n_test = 142  R2 = 0.6908  MAE = 27.31
骨架分组 fold 5: n_test = 142  R2 = 0.7481  MAE = 25.55
骨架分组 fold 6: n_test = 142  R2 = 0.6946  MAE = 31.24
骨架分组 fold 7: n_test = 142  R2 = 0.6585  MAE = 33.09
骨架分组 fold 8: n_test = 141  R2 = 0.6950  MAE = 27.14
骨架分组 fold 9: n_test = 141  R2 = 0.5804  MAE = 29.52
骨架分组10折: R2 = 0.6104 +- 0.1376
